# Hands-on Task: Build a Minimal RAG Pipeline

Embeds a small set of documents, retrieves the closest one to a question, and answers using only that context. Swap the sample docs for your own.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai numpy

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

## Option A: Use the Sample Documents
Runs out of the box against Ojasa Mirai's docs. Skip this and run Option B instead to use your own.

In [ ]:
import os, urllib.request
filenames = ["01_refund.txt", "02_enrollment.txt", "03_certificate.txt", "04_mentors.txt", "05_access.txt"]
base_url = "https://raw.githubusercontent.com/Roopesht/codeexamples/main/genai/llm/704/docs/"
os.makedirs("docs", exist_ok=True)
for name in filenames:
    urllib.request.urlretrieve(base_url + name, f"docs/{name}")
docs = [open(f"docs/{name}").read() for name in filenames]

## Option B: Upload Your Own Documents
Colab can't see files on your computer by default — this opens a picker to upload 3-5 of your own .txt files, which then replace `docs`.

In [ ]:
from google.colab import files
uploaded = files.upload()
docs = [content.decode() for content in uploaded.values()]

In [ ]:
import numpy as np
def embed_docs(docs):
    return [client.embeddings.create(model="text-embedding-3-small", input=d).data[0].embedding for d in docs]
def retrieve(query, docs, doc_vectors):
    query_vector = client.embeddings.create(model="text-embedding-3-small", input=query).data[0].embedding
    scores = [np.dot(query_vector, v) / (np.linalg.norm(query_vector) * np.linalg.norm(v)) for v in doc_vectors]
    return docs[scores.index(max(scores))]
def answer(query, context):
    prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer using only the context above."
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

In [ ]:
question = "What's Ojasa Mirai's refund policy for its courses?"
doc_vectors = embed_docs(docs)
context = retrieve(question, docs, doc_vectors)
print("Retrieved:", context)
print("Answer:", answer(question, context))

## How to Extend
Ask a question none of your documents can answer and check the model admits it instead of guessing. Add a 6th unrelated document and confirm retrieval still picks the right one.

## Done When
The printed answer is correct, and only your documents could have produced it.